In [5]:
import math

In [6]:
from dataclasses import dataclass
import numpy as np

@dataclass
class CombinedResult:
    R_comb: float                 # combined central value
    weights: np.ndarray           # BLUE weights (sum to 1)
    sigma_stat: float             # absolute statistical uncertainty
    sigma_sys_unc: float          # absolute uncorrelated systematic
    sigma_sys_com: float          # absolute fully-correlated (multiplicative) systematic
    sigma_sys_tot: float          # sqrt(sys_unc^2 + sys_com^2)
    sigma_tot: float              # sqrt(stat^2 + sys_tot^2)

def _build_common_cov(R: np.ndarray, c_common):
    """
    Build the covariance matrix for fully-correlated multiplicative systematics.
    - If c_common is a scalar: uses same relative error for all points.
    - If c_common is a 1D array of length n: per-point relative errors (still fully correlated, rho=1).
    - If c_common is a 2D array/list with shape (k, n): k fully-correlated sources; sum their covariances.
    Returns an (n x n) covariance matrix C.
    """
    R = np.asarray(R, float).reshape(-1)
    n = R.size

    if c_common is None:
        return np.zeros((n, n), dtype=float)

    c_arr = np.asarray(c_common, dtype=float)

    # Case 1: scalar
    if c_arr.ndim == 0:
        v = R * float(c_arr)               # absolute fully-correlated error per point
        return np.outer(v, v)

    # Case 2: 1D vector of length n
    if c_arr.ndim == 1:
        if c_arr.size != n:
            raise ValueError("c_common vector must have same length as R")
        v = R * c_arr
        return np.outer(v, v)

    # Case 3: 2D array: rows are different fully-correlated sources
    if c_arr.ndim == 2:
        if c_arr.shape[1] != n:
            raise ValueError("c_common 2D must have shape (k, n)")
        C = np.zeros((n, n), dtype=float)
        for row in c_arr:
            v = R * row
            C += np.outer(v, v)
        return C

    raise ValueError("Unsupported shape for c_common")

def combine_ratio_with_correlated_norm(R, s_stat, u_unc, c_common=None, use_C_in_weights=False):
    """
    Combine multiple measurements of R with:
      - uncorrelated statistical relative errors s_stat (array-like)
      - uncorrelated systematic relative errors u_unc (array-like)
      - fully-correlated *multiplicative* relative error(s) c_common:
          * scalar: same for all points
          * 1D array (len n): per-point relative but fully correlated (rho=1)
          * 2D array (k x n): k fully-correlated sources; summed

    Returns a CombinedResult with full stat/syst breakdown.
    """
    R = np.asarray(R, dtype=float).reshape(-1)
    s = np.asarray(s_stat, dtype=float).reshape(-1)
    u = np.asarray(u_unc, dtype=float).reshape(-1)
    n = R.size
    if not (s.size == n and u.size == n):
        raise ValueError("R, s_stat, u_unc must have the same length")

    # Absolute uncorrelated pieces
    sig_stat = s * R
    sig_unc  = u * R
    S = np.diag(sig_stat**2)
    U = np.diag(sig_unc**2)

    # Fully-correlated multiplicative piece(s)
    C = _build_common_cov(R, c_common)

    # Full covariance
    # V = S + U + C
    V = S + U if not use_C_in_weights else (S + U + C)

    # BLUE weights
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # Combined central value
    R_comb = float(w @ R)

    # Error decomposition with same weights
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ C @ w)   # if c_common is a vector: equals (sum_i w_i R_i c_i)^2

    sigma_stat = np.sqrt(var_stat)
    sigma_sys_unc = np.sqrt(var_unc)
    sigma_sys_com = np.sqrt(var_com)
    sigma_sys_tot = np.sqrt(var_unc + var_com)
    sigma_tot = np.sqrt(var_stat + var_unc + var_com)

    return CombinedResult(
        R_comb=R_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_sys_unc=sigma_sys_unc,
        sigma_sys_com=sigma_sys_com,
        sigma_sys_tot=sigma_sys_tot,
        sigma_tot=sigma_tot,
    )




In [7]:
def decompose_common_sources(R, weights, c_common, source_labels=None):
    """
    Decompose sigma_sys_com into contributions from each fully-correlated source.

    R            : array of central values (length n)
    weights      : BLUE weights used for the final combination (length n)
    c_common     : (k x n) array (or list of lists), each row = per-point *relative* error of source k
                   (still fully correlated, rho=1 for every source)
    source_labels: optional list of k labels (strings)

    Returns:
      sigmas_k      : array (k,) of absolute sigma from each source (std dev, not variance)
      sigma_com_tot : scalar, sqrt(sum_k sigmas_k^2), should match resB.sigma_sys_com
      fracs_k       : array (k,) of variance fractions (sigmas_k^2 / sum sigmas_k^2)
    """
    R = np.asarray(R, float).reshape(-1)
    w = np.asarray(weights, float).reshape(-1)
    C = np.asarray(c_common, float)

    if C.ndim == 1:
        C = C[None, :]  # make it (1, n)

    if C.shape[1] != R.size:
        raise ValueError("c_common must have shape (k, n) where n == len(R)")

    # For a fully-correlated multiplicative source k:
    # variance contribution = ( sum_i w_i * R_i * c_{k,i} )^2
    # -> std dev contribution = abs( sum_i w_i * R_i * c_{k,i} )
    t = w * R                      # length-n helper
    sigmas_k = np.abs(C @ t)       # length-k
    var_k = sigmas_k**2
    sigma_com_tot = float(np.sqrt(np.sum(var_k)))
    fracs_k = var_k / np.sum(var_k)

    # Pretty print
    if source_labels is None:
        source_labels = [f"source #{i+1}" for i in range(C.shape[0])]
    print("== Fully-correlated (multiplicative) source breakdown ==")
    for lbl, sig_k, frac in zip(source_labels, sigmas_k, fracs_k):
        print(f"  {lbl:>12s}: sigma = {sig_k:.12e}   (variance share {frac*100:6.2f}%)")
    print(f"  --> Quadrature sum (should match sigma_sys_com): {sigma_com_tot:.12e}")
    return sigmas_k, sigma_com_tot, fracs_k

In [11]:
math.sqrt(1.928 ** 2 + 0.342 ** 2 + 0.018 ** 2)

1.9581807883849742

In [12]:
math.sqrt(1.298 ** 2 + 1.340 ** 2)

1.8655840908412573

In [13]:
math.sqrt(1.149 ** 2 + 0.220 ** 2 + 0.008 ** 2)

1.1698995683390947

In [14]:
math.sqrt(0.929 ** 2 + 0.484 ** 2)

1.0475194508933952

In [17]:
# ---------------------------
# Br: D+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.1765e-04, 1.2689e-04]
s  = [5.9817e-06/R[0], 6.2142e-06/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.958**2 + 1.866**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.170**2 + 1.048**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["Frist common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.48792616 0.51207384]
  R_comb  = 1.223815623225e-04
  sigma_stat   = 4.317908696375e-06
  sigma_sys_unc= 1.949862003398e-06
  sigma_sys_com= 2.967296129482e-06
  sigma_sys_tot= 3.550606730171e-06
  sigma_stat_sys_tot= 5.590272235103e-06


== Fully-correlated (multiplicative) source breakdown ==
  Frist common: sigma = 5.207274429041e-07   (variance share   3.08%)
  Second common: sigma = 2.921247892639e-06   (variance share  96.92%)
  --> Quadrature sum (should match sigma_sys_com): 2.967296129482e-06

Cross-check:
  sigma_sys_com (from resB) = 2.967296129482e-06
  sigma_sys_com (decomposed sum) = 2.967296129482e-06


In [18]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")

# print(f"x1 = {R[0]:.8e} pm (stat) pm (sys)")

Final results:
R = 1.223815623225e-04 pm 4.317908696375e-06 (stat) pm 3.550606730171e-06 (total sys)

R = 1.223815623225e-04 pm 4.317908696375e-06 (stat) pm 2.018196943336e-06 (sys without norm Br) pm 2.921247892639e-06 (sys norm Br)

R1 = 1.176500000000e-04 pm 5.981700000000e-06 (stat) pm 4.365587267463e-06 (total sys)
R2 = 1.268900000000e-04 pm 6.214200000000e-06 (stat) pm 3.748413288069e-06 (total sys)

R1 = 1.176500000000e-04 pm 5.981700000000e-06 (stat) pm 3.342420142427e-06 (sys without norm Br) pm 2.808305500000e-06 (sys norm Br)
R2 = 1.268900000000e-04 pm 6.214200000000e-06 (stat) pm 2.208298718552e-06 (sys without norm Br) pm 3.028864300000e-06 (sys norm Br)


In [20]:
# ---------------------------
# Two measurements 
R  = [3.1207e-02, 3.3658e-02]
s  = [1.5867e-03/R[0], 1.6483e-03/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.958**2 + 1.866**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.170**2 + 1.048**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    # [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.48074163 0.51925837]
  R_comb  = 3.247970227326e-02
  sigma_stat   = 1.146475720416e-03
  sigma_sys_unc= 5.143299889012e-04
  sigma_sys_com= 1.382367102127e-04
  sigma_sys_tot= 5.325830691390e-04
  sigma_stat_sys_tot= 1.264140539275e-03


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 1.382367102127e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 1.382367102127e-04

Cross-check:
  sigma_sys_com (from resB) = 1.382367102127e-04
  sigma_sys_com (decomposed sum) = 1.382367102127e-04


In [21]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

Final results:
R = 3.247970227326e-02 pm 1.146475720416e-03 (stat) pm 5.325830691390e-04 (total sys)

R1 = 3.120700000000e-02 pm 1.586700000000e-03 (stat) pm 8.865865311069e-04 (total sys)
R2 = 3.365800000000e-02 pm 1.648300000000e-03 (stat) pm 5.857586749864e-04 (total sys)


In [22]:
math.sqrt(1.472 ** 2 + 0.047 ** 2 + 0.058 ** 2)

1.4738917870725787

In [24]:
math.sqrt(1.245 ** 2 + 0.787 ** 2 )

1.4728862821005566

In [25]:
math.sqrt(1.102 ** 2 + 1.167 ** 2 + 0.053 ** 2)

1.605958280902714

In [26]:
math.sqrt(1.351 ** 2 + 0.227 ** 2 )

1.3699379547994135

In [27]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.5930e-03, 1.5952e-03]
s  = [1.9557e-05/R[0], 2.1910e-05/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.474**2 + 1.473**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 1.606**2 + 1.370**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")




Two fully-correlated sources summed:
  weights = [0.50731106 0.49268894]
  R_comb  = 1.594083915674e-03
  sigma_stat   = 1.466164510295e-05
  sigma_sys_unc= 2.477389103217e-05
  sigma_sys_com= 2.633880495652e-05
  sigma_sys_tot= 3.615906972547e-05
  sigma_stat_sys_tot= 3.901848485700e-05


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 6.511277567703e-06   (variance share   6.11%)
  Second common: sigma = 2.552128348994e-05   (variance share  93.89%)
  --> Quadrature sum (should match sigma_sys_com): 2.633880495652e-05

Cross-check:
  sigma_sys_com (from resB) = 2.633880495652e-05
  sigma_sys_com (decomposed sum) = 2.633880495652e-05


In [28]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")


Final results:
R = 1.594083915674e-03 pm 1.466164510295e-05 (stat) pm 3.615906972547e-05 (total sys)

R = 1.594083915674e-03 pm 1.466164510295e-05 (stat) pm 2.561527693462e-05 (sys without norm Br) pm 2.552128348994e-05 (sys norm Br)

R1 = 1.593000000000e-03 pm 1.955700000000e-05 (stat) pm 4.409955735280e-05 (total sys)
R2 = 1.595200000000e-03 pm 2.191000000000e-05 (stat) pm 4.353938774799e-05 (total sys)

R1 = 1.593000000000e-03 pm 1.955700000000e-05 (stat) pm 3.597666623338e-05 (sys without norm Br) pm 2.550393000000e-05 (sys norm Br)
R2 = 1.595200000000e-03 pm 2.191000000000e-05 (stat) pm 3.526230282597e-05 (sys without norm Br) pm 2.553915200000e-05 (sys norm Br)


In [29]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [9.3708e-02,  9.3836e-02]
s  = [1.1504e-03/R[0], 1.2888e-03/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.474**2 + 1.473**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 1.606**2 + 1.370**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    # [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "(1.60%, 1.60%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.50703236 0.49296764]
  R_comb  = 9.377109985770e-02
  sigma_stat   = 8.624847652710e-04
  sigma_sys_unc= 1.457281335118e-03
  sigma_sys_com= 3.830235709877e-04
  sigma_sys_tot= 1.506776674101e-03
  sigma_stat_sys_tot= 1.736161258622e-03


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 3.830235709877e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 3.830235709877e-04

Cross-check:
  sigma_sys_com (from resB) = 3.830235709877e-04
  sigma_sys_com (decomposed sum) = 3.830235709877e-04


In [30]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")


Final results:
R = 9.377109985770e-02 pm 8.624847652710e-04 (stat) pm 1.506776674101e-03 (total sys)

R1 = 9.370800000000e-02 pm 1.150400000000e-03 (stat) pm 2.116322309728e-03 (total sys)
R2 = 9.383600000000e-02 pm 1.288800000000e-03 (stat) pm 2.074268711120e-03 (total sys)


In [34]:
# from dataclasses import dataclass
# import math

# @dataclass
# class AcpCombineResult:
#     A_comb: float
#     w1: float
#     w2: float
#     sigma_stat: float
#     sigma_syst_unc: float
#     sigma_syst_com: float
#     sigma_syst_tot: float
#     sigma_tot: float

# def combine_acp_two(a, b, c, d, x, y, z):
#     """
#     Combine two Acp-like measurements with absolute errors:
#       result1 = a ± b_stat ± c_syst ± d_common
#       result2 = x ± y_stat ± z_syst ± d_common
#     d_common is a fully correlated additive systematic (same magnitude in both).
#     Returns central value and split errors.
#     """
#     u1_sq = b*b + c*c  # uncorrelated variance of #1
#     u2_sq = y*y + z*z  # uncorrelated variance of #2

#     # BLUE weights using only uncorrelated parts
#     w1 = u2_sq / (u1_sq + u2_sq)
#     w2 = 1.0 - w1

#     # Combined central value
#     A_comb = w1*a + w2*x

#     # Error components
#     sigma_stat = math.sqrt((w1*b)**2 + (w2*y)**2)
#     sigma_syst_unc = math.sqrt((w1*c)**2 + (w2*z)**2)
#     sigma_syst_com = d  # fully correlated additive -> unchanged
#     sigma_syst_tot = math.sqrt(sigma_syst_unc**2 + sigma_syst_com**2)
#     sigma_tot = math.sqrt(sigma_stat**2 + sigma_syst_tot**2)

#     return AcpCombineResult(
#         A_comb=A_comb, w1=w1, w2=w2,
#         sigma_stat=sigma_stat,
#         sigma_syst_unc=sigma_syst_unc,
#         sigma_syst_com=sigma_syst_com,
#         sigma_syst_tot=sigma_syst_tot,
#         sigma_tot=sigma_tot
#     )

# def print_acp_result(res: AcpCombineResult, label="Acp"):
#     """Pretty-print the combination result with a clear breakdown."""
#     # helper to format relative (%) if central value is non-zero
#     def rel(x):
#         return (x / res.A_comb * 100.0) if res.A_comb != 0 else float("nan")

#     print(f"== {label} combination ==")
#     print(f"Weights: w1 = {res.w1:.4f}, w2 = {res.w2:.4f}")
#     print(f"{label} = {res.A_comb:.6e}")
#     print(f"  stat         : {res.sigma_stat:.6e}  (rel {rel(res.sigma_stat):.3f}%)")
#     print(f"  syst (uncorr): {res.sigma_syst_unc:.6e}  (rel {rel(res.sigma_syst_unc):.3f}%)")
#     print(f"  syst (common): {res.sigma_syst_com:.6e}  (rel {rel(res.sigma_syst_com):.3f}%)")
#     print(f"  syst (total) : {res.sigma_syst_tot:.6e}  (rel {rel(res.sigma_syst_tot):.3f}%)")
#     print(f"  TOTAL        : {res.sigma_tot:.6e}     (rel {rel(res.sigma_tot):.3f}%)")
#     print()
#     # compact “paper-style” line
#     print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
#           f"± {res.sigma_syst_unc:.6e} (syst-unc) ± {res.sigma_syst_com:.6e} (syst-com))")
#     print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
#           f"± {res.sigma_syst_tot:.6e} (syst-unc-tot)")
#     print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_tot:.6e})  [total]")




In [35]:
# # Br: D+ -> eta pi+
# a, b, c, d =  0.33e-2, 0.73e-2, math.sqrt(0.013**2 + 0.001**2 + 0.006**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
# x, y, z     =  0.12e-2, 0.90e-2, math.sqrt(0.013**2 + 0.007**2 + 0.001**2)*1e-2         # result2: x ± y (± z ± d)

# res = combine_acp_two(a, b, c, d, x, y, z)
# print_acp_result(res, label="A_CP")

In [36]:
# # Br: Ds+ -> eta pi+
# if __name__ == "__main__":
#     a, b, c, d =  0.10e-2, 0.46e-2, math.sqrt(0.009**2 + 0.001**2 + 0.016**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
#     x, y, z     =  -0.10e-2, 0.59e-2, math.sqrt(0.008**2 + 0.007**2 + 0.009**2)*1e-2         # result2: x ± y (± z ± d)

#     res = combine_acp_two(a, b, c, d, x, y, z)
#     print_acp_result(res, label="A_CP")

In [37]:
# # Br: D+ -> eta K+
# if __name__ == "__main__":
#     a, b, c, d =  9.25e-2, 7.93e-2, math.sqrt(0.158**2 + 0.012**2 + 0.005**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
#     x, y, z     =  2.46e-2, 8.64e-2, math.sqrt(0.152**2 + 0.004**2 + 0.052**2)*1e-2         # result2: x ± y (± z ± d)

#     res = combine_acp_two(a, b, c, d, x, y, z)
#     print_acp_result(res, label="A_CP")

In [38]:
# # Br: Ds+ -> eta K+
# if __name__ == "__main__":
#     a, b, c, d =  0.e-2, 0.e-2, math.sqrt()*1e-2, e-5   # result1: a ± b (± c ± d)
#     x, y, z     =  0.e-2, 0.e-2, math.sqrt()*1e-2         # result2: x ± y (± z ± d)

#     res = combine_acp_two(a, b, c, d, x, y, z)
#     print_acp_result(res, label="A_CP")

In [39]:
from dataclasses import dataclass
import numpy as np

@dataclass
class AcpCombineResult:
    A_comb: float              # combined central value
    weights: np.ndarray        # BLUE weights (sum to 1)
    sigma_stat: float          # absolute statistical uncertainty
    sigma_syst_unc: float      # absolute uncorrelated systematic
    sigma_syst_com: float      # absolute fully-correlated additive systematic
    sigma_syst_tot: float      # sqrt(syst_unc^2 + syst_com^2)
    sigma_tot: float           # sqrt(stat^2 + syst_tot^2)

def _outer_sum(v_list):
    """Sum of outer products: sum_k v_k v_k^T."""
    if not v_list:
        return None
    C = np.zeros((v_list[0].size, v_list[0].size), dtype=float)
    for v in v_list:
        C += np.outer(v, v)
    return C

def build_cov_additive_common(d_common, n):
    """
    Build K for fully-correlated *additive* systematics with possibly different magnitudes.
    - d_common: None | scalar | 1D (n,) | 2D (k,n)
      values are absolute sigmas (same unit as the measurement).
    """
    if d_common is None:
        return np.zeros((n, n), dtype=float)

    d = np.asarray(d_common, dtype=float)
    if d.ndim == 0:            # scalar -> same absolute size for all points
        v = np.full(n, float(d), dtype=float)
        return np.outer(v, v)
    if d.ndim == 1:            # per-point absolute sizes
        if d.size != n:
            raise ValueError("d_common vector must have length n")
        return np.outer(d, d)  # fully correlated -> rank-1
    if d.ndim == 2:            # multiple fully-correlated additive sources (independent)
        if d.shape[1] != n:
            raise ValueError("d_common 2D must have shape (k, n)")
        v_list = [row.astype(float) for row in d]
        return _outer_sum(v_list)
    raise ValueError("Unsupported shape for d_common")

def combine_acp_two_matrix(a, b_stat, c_unc, d_common,
                           x, y_stat, z_unc):
    """
    Combine two A_CP-like measurements using full matrix calculus.
      m1 = a ± b_stat (stat) ± c_unc (uncorr syst) ± d1 (common additive)
      m2 = x ± y_stat (stat) ± z_unc (uncorr syst) ± d2 (common additive)
    Here d_common can be:
      - scalar d  -> d1=d2=d
      - vector [d1, d2] -> different magnitudes but fully correlated (rho=1)
      - 2D (k,2) -> multiple independent additive common sources; each row is [d1_k, d2_k]
    Returns AcpCombineResult with BLUE weights and an error breakdown via quadratic forms.
    """
    # data vector
    y = np.array([a, x], dtype=float)
    n = y.size

    # diagonal pieces
    S = np.diag([b_stat**2, y_stat**2])    # stat (absolute)
    U = np.diag([c_unc**2,  z_unc**2 ])    # uncorrelated syst (absolute)

    # fully-correlated additive piece(s)
    K = build_cov_additive_common(d_common, n)

    # full covariance and GLS weights
    V = S + U + K
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # combined central value
    A_comb = float(w @ y)

    # error decomposition (all via matrices)
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ K @ w)

    sigma_stat    = np.sqrt(var_stat)
    sigma_syst_unc= np.sqrt(var_unc)
    sigma_syst_com= np.sqrt(var_com)
    sigma_syst_tot= np.sqrt(var_unc + var_com)
    sigma_tot     = np.sqrt(var_stat + var_unc + var_com)

    return AcpCombineResult(
        A_comb=A_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_syst_unc=sigma_syst_unc,
        sigma_syst_com=sigma_syst_com,
        sigma_syst_tot=sigma_syst_tot,
        sigma_tot=sigma_tot,
    )

# (Optional) Jacobian-based propagation check: M Vx M^T = V
def propagate_with_jacobian(b_stat, y_stat, c_unc, z_unc, d_common):
    """
    Build V via M Vx M^T with additive fully-correlated offsets:
      y = x + a * z,  z~N(0, tau^2=1),  a = [d1, d2]
    Returns V (should equal S+U+K).
    """
    S = np.diag([b_stat**2, y_stat**2])
    U = np.diag([c_unc**2,  z_unc**2 ])
    if d_common is None:
        a = np.zeros(2)
    else:
        d = np.asarray(d_common, float)
        if d.ndim == 0:
            a = np.array([float(d), float(d)], float)
        elif d.ndim == 1 and d.size == 2:
            a = d.astype(float)
        else:
            raise ValueError("Jacobian check supports scalar or 1D len-2 d_common only.")
    M  = np.column_stack([np.eye(2), a.reshape(2,1)])  # [ I | a ]
    Vx = np.zeros((3,3), float)
    Vx[:2,:2] = S + U          # diag for x1,x2
    Vx[2,2]   = 1.0            # Var(z)=tau^2
    V = M @ Vx @ M.T
    return V

def print_acp_result(res: AcpCombineResult, label="Acp"):
    """Pretty-print the combination result with a clear breakdown."""
    # helper to format relative (%) if central value is non-zero
    def rel(x):
        return (x / res.A_comb * 100.0) if res.A_comb != 0 else float("nan")

    print(f"== {label} combination ==")
    # print(f"Weights: w1 = {res.w1:.4f}, w2 = {res.w2:.4f}")
    print(f"Weights: {res.weights}")
    print(f"{label} = {res.A_comb:.6e}")
    print(f"  stat         : {res.sigma_stat:.6e}  (rel {rel(res.sigma_stat):.3f}%)")
    print(f"  syst (uncorr): {res.sigma_syst_unc:.6e}  (rel {rel(res.sigma_syst_unc):.3f}%)")
    print(f"  syst (common): {res.sigma_syst_com:.6e}  (rel {rel(res.sigma_syst_com):.3f}%)")
    print(f"  syst (total) : {res.sigma_syst_tot:.6e}  (rel {rel(res.sigma_syst_tot):.3f}%)")
    print(f"  TOTAL        : {res.sigma_tot:.6e}     (rel {rel(res.sigma_tot):.3f}%)")
    print()
    # compact “paper-style” line
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_unc:.6e} (syst-unc) ± {res.sigma_syst_com:.6e} (syst-com))")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_tot:.6e} (syst-unc-tot)")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_tot:.6e})  [total]")


In [28]:
# Acp: D+ -> eta pi+
a, b, c, d =  0.49389e-2, 0.35953e-2, math.sqrt(0.020**2 + 0.0086**2 + 0.000)*1e-2, [0.000068,0.000068]   # result1: a ± b (± c ± d)
x, y, z     =  -0.06098e-2, 0.44942e-2, math.sqrt(0.022**2 + 0.0092**2 + 0.0102**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
4.938900e-03 ± 3.595300e-03 ± 2.280789e-04
eta->pipipi
-6.098000e-04 ± 4.494200e-03 ± 2.681268e-04


== A_CP combination ==
Weights: [0.60968431 0.39031569]
A_CP = 2.773155e-03
  stat         : 2.807476e-03  (rel 101.238%)
  syst (uncorr): 1.669306e-04  (rel 6.020%)
  syst (common): 6.800000e-05  (rel 2.452%)
  syst (total) : 1.802494e-04  (rel 6.500%)
  TOTAL        : 2.813256e-03     (rel 101.446%)

A_CP = (2.773155e-03 ± 2.807476e-03 (stat) ± 1.669306e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (2.773155e-03 ± 2.807476e-03 (stat) ± 1.802494e-04 (syst-unc-tot)
A_CP = (2.773155e-03 ± 2.813256e-03)  [total]


In [29]:
# Acp: Ds+ -> eta pi+
a, b, c, d =  0.08267e-2, 0.23568e-2, math.sqrt(0.019**2 + 0.0086**2 + 0.0)*1e-2, [0.000068, 0.000068] # result1: a ± b (± c ± d)
x, y, z     =  -0.05604e-2, 0.30262e-2, math.sqrt(0.024**2 + 0.0092**2 + 0.0093**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
8.267000e-04 ± 2.356800e-03 ± 2.193627e-04
eta->pipipi
-5.604000e-04 ± 3.026200e-03 ± 2.816682e-04


== A_CP combination ==
Weights: [0.62253708 0.37746292]
A_CP = 3.031212e-04
  stat         : 1.859425e-03  (rel 613.426%)
  syst (uncorr): 1.658371e-04  (rel 54.710%)
  syst (common): 6.800000e-05  (rel 22.433%)
  syst (total) : 1.792372e-04  (rel 59.131%)
  TOTAL        : 1.868044e-03     (rel 616.270%)

A_CP = (3.031212e-04 ± 1.859425e-03 (stat) ± 1.658371e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (3.031212e-04 ± 1.859425e-03 (stat) ± 1.792372e-04 (syst-unc-tot)
A_CP = (3.031212e-04 ± 1.868044e-03)  [total]


In [41]:
math.sqrt(0.287 ** 2 + 0.292 **2 )

0.4094300917128588

In [42]:
math.sqrt(0.185 ** 2 + 0.136 **2 )

0.2296105398277701

In [43]:
math.sqrt(0.054 ** 2 + 0.113 **2 )

0.12523977004130918

In [44]:
math.sqrt(0.053 ** 2 + 0.104 **2 )

0.11672617529928751

In [45]:
# Acp: D+ -> eta K+
a, b, c, d =  9.42305e-2, 4.52576e-2, math.sqrt(0.409**2 + 0.0203**2 + 0.0893**2)*1e-2, [0.000067, 0.000067] # result1: a ± b (± c ± d)
x, y, z     =  0.31169e-2, 4.50164e-2, math.sqrt(0.230**2 + 0.0125**2 + 0.1845**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
9.423050e-02 ± 4.525760e-02 ± 4.191807e-03
eta->pipipi
3.116900e-03 ± 4.501640e-02 ± 2.951972e-03


== A_CP combination ==
Weights: [0.4962654 0.5037346]
A_CP = 4.833343e-02
  stat         : 3.191641e-02  (rel 66.034%)
  syst (uncorr): 2.556637e-03  (rel 5.290%)
  syst (common): 6.700000e-05  (rel 0.139%)
  syst (total) : 2.557514e-03  (rel 5.291%)
  TOTAL        : 3.201871e-02     (rel 66.245%)

A_CP = (4.833343e-02 ± 3.191641e-02 (stat) ± 2.556637e-03 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (4.833343e-02 ± 3.191641e-02 (stat) ± 2.557514e-03 (syst-unc-tot)
A_CP = (4.833343e-02 ± 3.201871e-02)  [total]


In [46]:
# Acp: Ds+ -> eta K+
a, b, c, d =  2.96231e-2, 1.09966e-2, math.sqrt(0.125**2 + 0.0203**2 + 0.0667**2)*1e-2, [0.000067, 0.000067]   # result1: a ± b (± c ± d)
x, y, z     =  0.83373e-2, 1.30885e-2, math.sqrt(0.117**2 + 0.0125**2 + 0.0736**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
2.962310e-02 ± 1.099660e-02 ± 1.432860e-03
eta->pipipi
8.337300e-03 ± 1.308850e-02 ± 1.389500e-03


== A_CP combination ==
Weights: [0.58484088 0.41515912]
A_CP = 2.078611e-02
  stat         : 8.419466e-03  (rel 40.505%)
  syst (uncorr): 1.016217e-03  (rel 4.889%)
  syst (common): 6.700000e-05  (rel 0.322%)
  syst (total) : 1.018424e-03  (rel 4.900%)
  TOTAL        : 8.480837e-03     (rel 40.801%)

A_CP = (2.078611e-02 ± 8.419466e-03 (stat) ± 1.016217e-03 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (2.078611e-02 ± 8.419466e-03 (stat) ± 1.018424e-03 (syst-unc-tot)
A_CP = (2.078611e-02 ± 8.480837e-03)  [total]


In [3]:
import math
run1 = 1/math.sqrt(428)
run2 = 1/math.sqrt(575.47)

In [7]:
(run1-run2)/run1

0.13759644042921565